## MODELS OPTIMIZATION AND MLFLOW

In [ ]:
import os
import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

# Fix: Dynamically resolve the absolute path to the root project directory where mlflow.db lives
current_dir = os.getcwd()
root_dir = os.path.abspath(os.path.join(current_dir, "..")) if "notebooks" in current_dir or "src" in current_dir else current_dir
db_path = os.path.join(root_dir, "mlflow.db")

# Setup MLflow tracking uri with the absolute SQLite path
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("Customer_Churn_Optimization")

# Load processed training and testing datasets
train_df = pd.read_csv("../data/processed/train_resampled_smote.csv")
test_df = pd.read_csv("../data/processed/test_data.csv")

X_train_resampled = train_df.drop(columns=["churn"])
y_train_resampled = train_df["churn"]
X_test = test_df.drop(columns=["churn"])
y_test = test_df["churn"]

# Define preprocessing parameters for tracking
preprocessing_params = {
    "imputation": "median_mode",
    "scaling": "StandardScaler",
    "encoding": "OneHotEncoder",
    "imbalance_handler": "SMOTE",
    "resampled_train_shape": str(X_train_resampled.shape)
}

# Run 1: Baseline Model Training and Logging
with mlflow.start_run(run_name="XGBoost_Baseline"):
    mlflow.set_tag("model_type", "XGBoost")
    mlflow.set_tag("phase", "baseline")
    mlflow.log_params(preprocessing_params)
    
    base_model = XGBClassifier(eval_metric="logloss", random_state=42)
    base_model.fit(X_train_resampled, y_train_resampled)
    
    y_pred_base = base_model.predict(X_test)
    y_prob_base = base_model.predict_proba(X_test)[:, 1]
    
    base_metrics = {
        "accuracy": accuracy_score(y_test, y_pred_base),
        "precision": precision_score(y_test, y_pred_base, zero_division=0),
        "recall": recall_score(y_test, y_pred_base, zero_division=0),
        "f1_score": f1_score(y_test, y_pred_base, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob_base)
    }
    
    mlflow.log_metrics(base_metrics)
    
    # Log baseline model with trusted types for skops
    mlflow.sklearn.log_model(
        base_model, 
        "baseline_churn_model",
        skops_trusted_types=["xgboost.sklearn.XGBClassifier", "xgboost.core.Booster"]
    )

# Define hyperparameters distribution for randomized search
param_distributions = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [3, 5, 7, 10],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "subsample": [0.8, 1.0]
}

# Configure Randomized Search Cross-Validation
random_search = RandomizedSearchCV(
    estimator=XGBClassifier(eval_metric="logloss", random_state=42),
    param_distributions=param_distributions,
    n_iter=5,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1
)

# Run 2: Hyperparameter Tuning, Evaluation, and Artifact Logging
with mlflow.start_run(run_name="XGBoost_Tuned_Optimization"):
    mlflow.set_tag("model_type", "XGBoost")
    mlflow.set_tag("phase", "hyperparameter_tuning")
    mlflow.log_params(preprocessing_params)
    
    random_search.fit(X_train_resampled, y_train_resampled)
    
    best_model = random_search.best_estimator_
    best_params = random_search.best_params_
    
    y_pred_tuned = best_model.predict(X_test)
    y_prob_tuned = best_model.predict_proba(X_test)[:, 1]
    
    tuned_metrics = {
        "accuracy": accuracy_score(y_test, y_pred_tuned),
        "precision": precision_score(y_test, y_pred_tuned, zero_division=0),
        "recall": recall_score(y_test, y_pred_tuned, zero_division=0),
        "f1_score": f1_score(y_test, y_pred_tuned, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob_tuned)
    }
    
    mlflow.log_params(best_params)
    mlflow.log_metrics(tuned_metrics)
    
    # Generate and save a confusion matrix plot artifact
    os.makedirs("../artifacts/figures/classification", exist_ok=True)
    cm = confusion_matrix(y_test, y_pred_tuned)
    plt.figure(figsize=(6, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
    plt.title("Confusion Matrix - Tuned XGBoost")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    cm_path = "../artifacts/figures/classification/confusion_matrix.png"
    plt.savefig(cm_path)
    plt.close()
    
    # Log the plot artifact to MLflow
    mlflow.log_artifact(cm_path)
    
    # Log the tuned model with trusted types for skops
    mlflow.sklearn.log_model(
        best_model, 
        "tuned_churn_model", 
        skops_trusted_types=["xgboost.sklearn.XGBClassifier", "xgboost.core.Booster"]
    )

# Save the final optimized model locally for Streamlit deployment
os.makedirs("../models/classification", exist_ok=True)
model_path = "../models/classification/churn_model.pkl"
joblib.dump(best_model, model_path)

2026/10/09 11:17:11 ERROR mlflow.store.db.utils: SQLAlchemy database error. The following exception is caught.
(raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)
(sqlite3.OperationalError) attempt to write a readonly database
[SQL: INSERT INTO runs (run_uuid, name, source_type, source_name, entry_point_name, user_id, status, start_time, end_time, deleted_time, source_version, lifecycle_stage, artifact_uri, experiment_id) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: ('78b8b2101402449d8409f9db21b69897', 'XGBoost_Baseline', 'UNKNOWN', '', '', 'ememes', 'RUNNING', 1791544631247, None, None, '', 'active', '/home/ememes/ai_projects/client_churn_prediction/notebooks/mlruns/1/78b8b2101402449d8409f9db21b69897/artifacts', '1')]
(Background on this error at: https://sqlalche.me/e/21/e3q8)
Traceback (most recent call last):
  File "/home/ememes/ai_projects/client_churn_prediction/venv/lib/python3.14

MlflowException: (raised as a result of Query-invoked autoflush; consider using a session.no_autoflush block if this flush is occurring prematurely)
(sqlite3.OperationalError) attempt to write a readonly database
[SQL: INSERT INTO runs (run_uuid, name, source_type, source_name, entry_point_name, user_id, status, start_time, end_time, deleted_time, source_version, lifecycle_stage, artifact_uri, experiment_id) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)]
[parameters: ('78b8b2101402449d8409f9db21b69897', 'XGBoost_Baseline', 'UNKNOWN', '', '', 'ememes', 'RUNNING', 1791544631247, None, None, '', 'active', '/home/ememes/ai_projects/client_churn_prediction/notebooks/mlruns/1/78b8b2101402449d8409f9db21b69897/artifacts', '1')]
(Background on this error at: https://sqlalche.me/e/21/e3q8)

# Phase 4: Model Optimization & MLflow Tracking - Summary

## Overview
In this final modeling notebook, we optimized our top-performing classification model (**XGBoost**) using hyperparameter tuning and tracked all experiment runs, preprocessing parameters, metrics, model artifacts (such as confusion matrices), and final model binaries using **MLflow** with a local SQLite backend (`mlflow.db`).

## Key Steps & Compliance Checklist
1. **Data Loading:** Loaded SMOTE-resampled training features and untouched test features.
2. **Preprocessing Parameters Tracking:** Explicitly logged preprocessing metadata (imputation, scaling, encoding, SMOTE configuration) inside MLflow runs.
3. **Baseline Run (`XGBoost_Baseline`):** Trained default parameters and logged evaluation metrics to establish a performance benchmark.
4. **Hyperparameter Optimization (`XGBoost_Tuned_Optimization`):** Applied `RandomizedSearchCV` across a parameter grid optimizing for the F1-score with 3-fold cross-validation.
5. **Artifact & Model Logging:** Generated and stored visual evaluation artifacts (Confusion Matrix) alongside model binaries and performance metrics.
6. **Model Persistence:** Successfully exported the final production-ready model to `../models/classification/churn_model.pkl` for Streamlit deployment.